<center>
    <p style="text-align:center">
        <img alt="phoenix logo" src="https://raw.githubusercontent.com/Arize-ai/phoenix-assets/9e6101d95936f4bd4d390efc9ce646dc6937fb2d/images/socal/github-large-banner-phoenix.jpg" width="1000"/>
        <br>
        <br>
        <a href="https://arize.com/docs/phoenix/">Docs</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://arize-ai.slack.com/join/shared_invite/zt-2w57bhem8-hq24MB6u7yE_ZF_ilOYSBw#/shared-invite/email">Community</a>
    </p>
</center>
<h1 align="center">Evals that Work</h1>

Building great AI native products requires a rigorous evaluation process. While the idea of evaluation-driven development may seem novel to some, it really is the scientific method in disguise.  Just as scientists meticulously record experiments and take detailed notes to advance their understanding, AI systems require rigorous observation through tracing, annotations, and experimentation to reach their full potential. The goal of AI-native products is to build tools that empower humans, and it requires careful human judgment to align AI with human preferences and values.

note: this is inspired by a tutorial originally authored by [Ankur Goyal](https://www.braintrust.dev/docs/cookbook/recipes/Text2SQL-Data)

 <p style="text-align:center">
  <img src="https://storage.googleapis.com/arize-phoenix-assets/assets/images/scientific_method.png" width="60%" style="float: left">
  <img alt="AI dev as scientific method" src="https://storage.googleapis.com/arize-phoenix-assets/assets/gifs/20250524_1125_Forest%20Robots%20Interaction_simple_compose_01jw1n770bep1a829kw3cvvcsc.gif" width="40%" style="float: right"/>
</p>

In [1]:
# !pip install "arize-phoenix>=10.0.0" openai 'httpx<0.28' duckdb datasets pyarrow "pydantic>=2.0.0" nest_asyncio openinference-instrumentation-openai --quiet

In [2]:

# %%
# %pip install "arize-phoenix>=10.0.0" 'httpx<0.28' duckdb datasets pyarrow "pydantic>=2.0.0" nest_asyncio langchain-google-vertexai google-cloud-aiplatform --quiet


This tutorial assumes you have a locally running Phoenix server. We can think of phoenix like a video recorder, observing every activity of your AI application.

```shell
phoenix serve
```

Let's also setup tracing for OpenAI as we will be using their API to perform the synthesis.

In [3]:
from google.cloud.aiplatform import init as init_vertexai
PROJECT_ID = "seequent-labs-dev"  
LOCATION = "us-central1"  # Change to your desired location
init_vertexai(project=PROJECT_ID, location=LOCATION)

In [4]:
import os

os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "http://localhost:6006"


# %%
from phoenix.otel import register

tracer_provider = register(
    project_name="basketball-agent-test",
    auto_instrument=True,
)

tracer = tracer_provider.get_tracer(__name__)


  # Import the automatic instrumentor from OpenInference
from openinference.instrumentation.langchain import LangChainInstrumentor

# Finish automatic instrumentation
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

## Global session tracking (notebook-wide)
from openinference.instrumentation import using_session
import uuid

# Set this once at the top of your notebook
GLOBAL_SESSION_ID = str(uuid.uuid4())

def setup_session_context():
    """Call this once to set up global session tracking"""
    return using_session(session_id=GLOBAL_SESSION_ID)

# Set up the session context
session_context = setup_session_context()
session_context.__enter__()


## Decorator approach on every graph invocation:
# from openinference.instrumentation import using_session
# import uuid

# # Session setup - run once
# SESSION_ID = str(uuid.uuid4())
# THREAD_ID = str(uuid.uuid4())

# @using_session(session_id=SESSION_ID)
# def invoke(messages, thread_id=None):
#     """Simple wrapper for graph.invoke with automatic session tracking"""
#     config = {"configurable": {"thread_id": thread_id or THREAD_ID}}
#     return graph.invoke(messages, config=config)

# # Usage in any cell becomes very simple:
# result = invoke({"messages": [{"role": "user", "content": "Hello"}]})


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Attempting to instrument while already instrumented


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: basketball-agent-test
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {'user-agent': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



## Setting traces on functions

In [5]:
from phoenix.otel import register

tracer_provider = register(
    project_name="basketball-agent-test",
    auto_instrument=True,
)

tracer = tracer_provider.get_tracer(__name__)


@tracer.tool
def vector_search_function(query: str) -> list:
    """This will automatically create a TOOL span with name 'vector_search_function'"""
    
    return query.split(" ")  # Simulating a vector search by splitting the query into words

## Use a decorator factory to add custom attributes to the tool span.
## This allows us to import the decorator and set a category in the decorator, without having to modify the function itself.
from functools import wraps
from opentelemetry import trace
def custom_tracer_tool(**custom_attrs):
    """Decorator factory that combines @tracer.tool with custom attributes"""
    def decorator(func):
        @tracer.tool
        @wraps(func)
        def wrapper(*args, **kwargs):
            span = trace.get_current_span()
            # Add the custom attributes
            for key, value in custom_attrs.items():
                span.set_attribute(key, value)
            return func(*args, **kwargs)
        return wrapper
    return decorator

# Usage with the decorator factory
@custom_tracer_tool(custom_function="custom_function", category="data_processing")
def my_function():
    # Your function logic
    return "result"


Overriding of current TracerProvider is not allowed
Attempting to instrument while already instrumented


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: basketball-agent-test
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {'user-agent': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [6]:
vector_search_function('abcdef')

['abcdef']

In [7]:
my_function()

'result'

Then use this to search for it in the ui: category == "data_processing"

In [8]:


# %%
import nest_asyncio
nest_asyncio.apply()

# %% [markdown]
# ## Download Data

# %%
import duckdb
from datasets import load_dataset

data = load_dataset("suzyanil/nba-data")["train"]

conn = duckdb.connect(database=":memory:", read_only=False)
conn.register("nba", data.to_pandas())

conn.query("SELECT * FROM nba LIMIT 5").to_df().to_dict(orient="records")[0]

# %% [markdown]
# ## Implement Text2SQL with Vertex AI and LangChain

# %%
from langchain_google_vertexai import ChatVertexAI
from langchain_core.prompts import ChatPromptTemplate
from phoenix.client import Client
from phoenix.client.types import PromptVersion

phoenix_client = Client()

# Initialize Vertex AI model
llm = ChatVertexAI(
    model_name="gemini-1.5-pro",
    project=PROJECT_ID,
    location=LOCATION,
    temperature=0
)

columns = conn.query("DESCRIBE nba").to_df().to_dict(orient="records")

TASK_MODEL = "gemini-1.5-pro"
CONFIG = {"model": TASK_MODEL}

system_prompt = (
    "You are a SQL expert, and you are given a single table named nba with the following columns:\n"
    f'{",".join(column["column_name"] + ": " + column["column_type"] for column in columns)}\n'
    "Write a SQL query corresponding to the user's request. Return just the query text, "
    "with no formatting (backticks, markdown, etc.)."
)

# Create LangChain prompt template
prompt_template_lc = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", "{question}")
])

# Create Phoenix prompt template for tracking
prompt_template = phoenix_client.prompts.create(
    name="text2sql",
    version=PromptVersion(
        [
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": "{{question}}",
            },
        ],
        description="Initial prompt for text2sql",
        model_name=TASK_MODEL,
    ),
)

@tracer.chain
async def generate_query(question):
    # Use LangChain with Vertex AI
    chain = prompt_template_lc | llm
    response = await chain.ainvoke({"question": question})
    return response.content

# %%
query = await generate_query("Who won the most games?")
print(query)

# %%
@tracer.tool
def execute_query(query):
    return conn.query(query).fetchdf().to_dict(orient="records")

execute_query(query)

# %%
@tracer.chain
async def text2sql(question):
    query = await generate_query(question)
    results = None
    error = None
    try:
        results = execute_query(query)
    except duckdb.Error as e:
        error = str(e)

    return {
        "query": query,
        "results": results,
        "error": error,
    }

synthesis_system_prompt = """
You are a helpful assistant that can answer questions about the NBA. Answer the question based on the sql results.

Do not use sql or abbreviations for teams. Use the full team names and use an informative, concise voice.
Your response should be purely in natural language, do not include any sql or other technical details.

If the sql results are empty, say you don't know.
"""

# Create synthesis prompt template
synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system", synthesis_system_prompt),
    ("user", "The sql results of the query are: {results}. Answer the following question: {question}. Answer:")
])

@tracer.agent
async def basketball_agent(question):
    sql_response = await text2sql(question)
    if sql_response["error"]:
        raise Exception(sql_response["error"])
    
    results = sql_response["results"]
    
    # Use LangChain for synthesis
    synthesis_chain = synthesis_prompt | llm
    answer = await synthesis_chain.ainvoke({
        "results": results,
        "question": question
    })
    
    return answer.content

await basketball_agent("Who won the most games?")

# %%
from phoenix.trace import using_project

questions = [
    "Which team won the most games?",
    "Which team won the most games in 2015?",
    "Who led the league in 3 point shots?",
    "Which team had the biggest difference in records across two consecutive years?",
    "What is the average number of free throws per year?",
]

with using_project(project_name="basketball-agent-test"):
    for question in questions:
        try:
            answer = await basketball_agent(question)
            print(answer)
        except Exception as e:
            print(e)


SELECT Team FROM nba GROUP BY Team ORDER BY SUM(CASE WHEN WINorLOSS = 'W' THEN 1 ELSE 0 END) DESC LIMIT 1

The Golden State Warriors won the most games.

I'm sorry, I don't have the information to determine which team won the most games in 2015.

Catalog Error: Scalar Function with name `__postfix does not exist!
Did you mean "!__postfix"?
The Houston Rockets had the biggest difference in records across two consecutive years. This occurred between the 2011-12 season and the 2012-13 season.

The average number of free throws attempted per game across all the teams varies from about 12.7 to 20.7.



In [9]:

# %%
from phoenix.client import Client
from phoenix.client.types.spans import SpanQuery

phoenix_client = Client()
query = SpanQuery().where("name == 'basketball_agent'")

spans_df = phoenix_client.spans.get_spans_dataframe(
    project_identifier="basketball-agent-test", query=query
)
annotations_df = phoenix_client.spans.get_span_annotations_dataframe(
    spans_dataframe=spans_df, project_identifier="basketball-agent-test"
)

combined_df = annotations_df.join(spans_df, how="inner")
combined_df.head()


,name,span_kind,parent_id,start_time,end_time,status_code,status_message,events,context.span_id,context.trace_id,attributes.output.mime_type,attributes.input.value,attributes.input.mime_type,attributes.session.id,attributes.output.value,attributes.openinference.span.kind


In [10]:
annotations_df

""


In [11]:
spans_df

,name,span_kind,parent_id,start_time,end_time,status_code,status_message,events,context.span_id,context.trace_id,attributes.output.mime_type,attributes.input.value,attributes.input.mime_type,attributes.session.id,attributes.output.value,attributes.openinference.span.kind
context.span_id,,,,,,,,,,,,,,,,
0ce3287f84e5a523,basketball_agent,AGENT,None,2025-07-02 23:00:34.497447+00:00,2025-07-02 23:00:36.033897+00:00,OK,,[],0ce3287f84e5a523,cb785e6ef68b4f3e19f03a2dfad6a328,text/plain,Who won the most games?,text/plain,e5d52196-3c70-4c33-9e3b-ea0876915fd0,The Golden State Warriors won the most games.\n,AGENT
d40a5c9c7f6a53b0,basketball_agent,AGENT,None,2025-07-02 23:00:36.036691+00:00,2025-07-02 23:00:37.494635+00:00,OK,,[],d40a5c9c7f6a53b0,5f9416129b1afadea586d88136d5eca8,text/plain,Which team won the most games?,text/plain,e5d52196-3c70-4c33-9e3b-ea0876915fd0,The Golden State Warriors won the most games.\n,AGENT
4dba918d4d3767a5,basketball_agent,AGENT,None,2025-07-02 23:00:37.498444+00:00,2025-07-02 23:00:39.494641+00:00,OK,,[],4dba918d4d3767a5,6f486ca9486a8b4f04237335e09c94b2,text/plain,Which team won the most games in 2015?,text/plain,e5d52196-3c70-4c33-9e3b-ea0876915fd0,"I'm sorry, I don't have the information to det...",AGENT
aa481fed895c712b,basketball_agent,AGENT,None,2025-07-02 23:00:39.501616+00:00,2025-07-02 23:00:40.664383+00:00,ERROR,Exception: Catalog Error: Scalar Function with...,"[{'name': 'exception', 'timestamp': '2025-07-0...",aa481fed895c712b,2cdc6302d67f97d97b9fab416b58c92b,None,Who led the league in 3 point shots?,text/plain,e5d52196-3c70-4c33-9e3b-ea0876915fd0,None,AGENT
85e40bd01f65ff0a,basketball_agent,AGENT,None,2025-07-02 23:00:40.666255+00:00,2025-07-02 23:00:45.170168+00:00,OK,,[],85e40bd01f65ff0a,33d42d6ca1d31e3391f2f05178c2bf8c,text/plain,Which team had the biggest difference in recor...,text/plain,e5d52196-3c70-4c33-9e3b-ea0876915fd0,The Houston Rockets had the biggest difference...,AGENT
264832c6cf890a3e,basketball_agent,AGENT,None,2025-07-02 23:00:45.174053+00:00,2025-07-02 23:00:49.742708+00:00,OK,,[],264832c6cf890a3e,f401572b6e425b2465459615986613e0,text/plain,What is the average number of free throws per ...,text/plain,e5d52196-3c70-4c33-9e3b-ea0876915fd0,The average number of free throws attempted pe...,AGENT
1b3251d9eacb28e8,basketball_agent,AGENT,None,2025-07-03 16:07:28.985934+00:00,2025-07-03 16:07:33.179104+00:00,OK,,[],1b3251d9eacb28e8,bcd9450cdc05805e7d260cb2a24a1a50,text/plain,Who won the most games?,text/plain,d44f9fed-85e4-4cc3-b9f2-25e73cd94fb6,The Golden State Warriors won the most games.\n,AGENT
23adc9b28609e8f9,basketball_agent,AGENT,None,2025-07-03 16:07:33.182255+00:00,2025-07-03 16:07:38.609588+00:00,OK,,[],23adc9b28609e8f9,4e65eec6ea6b5847ab40b097a55a67ac,text/plain,Which team won the most games?,text/plain,d44f9fed-85e4-4cc3-b9f2-25e73cd94fb6,The Golden State Warriors won the most games.\n,AGENT
8c4f777cf190aac2,basketball_agent,AGENT,None,2025-07-03 16:07:38.613221+00:00,2025-07-03 16:07:44.239991+00:00,OK,,[],8c4f777cf190aac2,1a8fc0767ae5e0fd2973776a17e1979d,text/plain,Which team won the most games in 2015?,text/plain,d44f9fed-85e4-4cc3-b9f2-25e73cd94fb6,"I'm sorry, I don't have the information to det...",AGENT


In [12]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [ ]:

# %%
examples_df = combined_df[
    ["annotation_name", "result.label", "attributes.input.value", "attributes.output.value"]
].head()
examples_df

# %%
eval_prompt = f"""
You are an expert evaluator of question and answer pairs. You will be given a human question and an answer from a model.
Your job is to determine if the answer is "correct" or "incorrect".

Here are some examples of correct and incorrect answers:
{'\n\n'.join([f"Question: {example['attributes.input.value']}\nAnswer: {example['attributes.output.value']}\nLabel: {example['result.label']}" for example in examples_df.to_dict(orient="records")])}

## Evaluation
Provide your answer in the following format:
Question: <question>
Answer: <answer>
Explanation: <explanation>
Label: <correct|incorrect>

Question: {{attributes.input.value}}
Answer: {{attributes.output.value}}
Explanation:
"""

print(eval_prompt)


KeyError: "['annotation_name', 'result.label'] not in index"

In [ ]:

# %%
from phoenix.evals import llm_classify
from phoenix.evals.models import VertexAIModel
from phoenix.evals.templates import PromptTemplate

# Use Vertex AI model for evaluations
evals_df = llm_classify(
    data=spans_df,
    model=VertexAIModel(
        model="gemini-1.5-pro",
        project=PROJECT_ID,
        location=LOCATION
    ),
    rails=["correct", "incorrect"],
    template=PromptTemplate(
        template=eval_prompt,
    ),
    exit_on_error=False,
    provide_explanation=True,
)

evals_df["score"] = evals_df["label"].apply(lambda x: 1 if x == "correct" else 0)
evals_df[["label", "score", "explanation"]].head()

# %%
import phoenix as px
from phoenix.trace import SpanEvaluations

px.Client().log_evaluations(
    SpanEvaluations(
        dataframe=evals_df,
        eval_name="llm_correctness",
    )
)

# %% [markdown]
# ## Experimentation

# %%
import pandas as pd
import phoenix as px

ds = px.Client().upload_dataset(
    dataset_name="nba-questions",
    dataframe=pd.DataFrame([{"question": question} for question in questions]),
    input_keys=["question"],
    output_keys=[],
)

# %%
def no_error(output):
    return 1.0 if output.get("error") is None else 0.0

def has_results(output):
    results = output.get("results")
    has_results = results is not None and len(results) > 0
    return 1.0 if has_results else 0.0

# %%
from phoenix.experiments import run_experiment

def task(input):
    return text2sql(input["question"])

experiment = run_experiment(
    ds,
    task=task,
    evaluators=[no_error, has_results],
    experiment_metadata=CONFIG,
    experiment_name="baseline",
)

# %% [markdown]
# ## Improved Prompt with Few-Shot Examples

# %%
samples = conn.query("SELECT * FROM nba LIMIT 1").to_df().to_dict(orient="records")[0]
sample_rows = "\n".join(
    f"{column['column_name']} | {column['column_type']} | {samples[column['column_name']]}"
    for column in columns
)

improved_system_prompt = (
    "You are a SQL expert, and you are given a single table named nba with the following columns:\n\n"
    "Column | Type | Example\n"
    "-------|------|--------\n"
    f"{sample_rows}\n"
    "\n"
    "Write a DuckDB SQL query corresponding to the user's request. "
    "Return just the query text, with no formatting (backticks, markdown, etc.)."
)

# Update LangChain prompt template
prompt_template_lc = ChatPromptTemplate.from_messages([
    ("system", improved_system_prompt),
    ("user", "{question}")
])

# Update Phoenix prompt template
prompt_template = phoenix_client.prompts.create(
    name="text2sql",
    version=PromptVersion(
        [
            {
                "role": "system",
                "content": improved_system_prompt,
            },
            {
                "role": "user",
                "content": "{{question}}",
            },
        ],
        description="Add few shot examples to the prompt",
        model_name=TASK_MODEL,
    ),
)

print(await generate_query("Which team won the most games in 2015?"))

# %%
experiment = run_experiment(
    ds,
    experiment_name="with examples",
    task=task,
    evaluators=[has_results, no_error],
    experiment_metadata=CONFIG,
)

# %%
from phoenix.evals.models import VertexAIModel
from phoenix.experiments import evaluate_experiment
from phoenix.experiments.evaluators.llm_evaluators import LLMCriteriaEvaluator

llm_evaluator = LLMCriteriaEvaluator(
    name="is_sql",
    criteria="is_sql",
    description="the output is a valid SQL query and that it executes without errors",
    model=VertexAIModel(
        model="gemini-1.5-pro",
        project=PROJECT_ID,
        location=LOCATION
    ),
)

evaluate_experiment(experiment, evaluators=[llm_evaluator])

# %% [markdown]
# ## Final Improved Agent

# %%
@tracer.agent
async def basketball_agent_improved(question):
    sql_response = await text2sql(question)
    if sql_response["error"]:
        raise Exception(sql_response["error"])
    
    results = sql_response["results"]
    
    improved_synthesis_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant that can answer questions about the NBA. Do not use sql or abbreviations for teams. Use the full team names and use an informative, concise voice. Your response should be purely in natural language, do not include any sql or other technical details."),
        ("user", "The sql results of the query are: {results}. Answer the following question: {question}.")
    ])
    
    synthesis_chain = improved_synthesis_prompt | llm
    answer = await synthesis_chain.ainvoke({
        "results": results,
        "question": question
    })
    
    return answer.content

with using_project(project_name="basketball-agent-improved"):
    for question in questions:
        try:
            answer = await basketball_agent_improved(question)
            print(answer)
        except Exception as e:
            print(e)

# %%
phoenix_client = Client()
query = SpanQuery().where("name == 'basketball_agent_improved'")

spans_df = phoenix_client.spans.get_spans_dataframe(
    project_identifier="basketball-agent-improved", query=query
)

# %%
evals_df = llm_classify(
    data=spans_df,
    model=VertexAIModel(
        model="gemini-1.5-pro",
        project=PROJECT_ID,
        location=LOCATION
    ),
    rails=["correct", "incorrect"],
    template=PromptTemplate(
        template=eval_prompt,
    ),
    exit_on_error=False,
    provide_explanation=True,
)

evals_df["score"] = evals_df["label"].apply(lambda x: 1 if x == "correct" else 0)
evals_df[["label", "score", "explanation"]].head()

# %%
px.Client().log_evaluations(
    SpanEvaluations(
        dataframe=evals_df,
        eval_name="llm_correctness",
    )
)
